In [1]:
import os
from pathlib import Path

# 항상 PRISM 루트로 고정
ROOT = Path('__file__').resolve().parent.parent if '__file__' in globals() else Path.cwd()
while ROOT.name != 'PRISM':
    ROOT = ROOT.parent
os.chdir(ROOT)

In [2]:
import pandas as pd

df = pd.read_csv('data/raw/fragella_raw.csv', low_memory=False)

In [3]:
# Confidence 결측 행 확인
df[df['Confidence'].isna()][['Name', 'Brand', 'Year', 'Gender', 'Popularity', 'Price']]

,Name,Brand,Year,Gender,Popularity,Price
17785,212,Carolina Herrera,NaN,men,Not popular,114.99
39351,212,Carolina Herrera,NaN,men,Not popular,114.99


In [4]:
# 두 행 동일 여부 검증
row1 = df[df['Confidence'].isna()].iloc[0]
row2 = df[df['Confidence'].isna()].iloc[1]

diff = [(col, row1[col], row2[col]) for col in df.columns if str(row1[col]) != str(row2[col])]

if diff:
    print('차이가 있는 컬럼:')
    for col, v1, v2 in diff:
        print(f'  {col}: {v1} vs {v2}')
else:
    print('두 행이 완전히 동일합니다.')

두 행이 완전히 동일합니다.


In [5]:
# 결측치 처리
print(f'처리 전: {len(df)}행')

# 필수 필드 결측 행 제거 (Confidence)
df = df[df['Confidence'].notna()].copy()
print(f'Confidence 결측 제거 후: {len(df)}행')

# 권장/선택 필드 결측 처리
df['Gender'] = df['Gender'].fillna('Unknown')
df['Price Value'] = df['Price Value'].fillna('Unknown')
df['Country'] = df['Country'].fillna('Unknown')
df['OilType'] = df['OilType'].fillna('Unknown')
# rating, Year, Price는 숫자형 컬럼이므로 NaN 유지 → UI 표시 단계에서 '-'로 처리
# Purchase URL은 선택 필드로 NaN 유지 → UI 표시 단계에서 미표시 처리
# Image Fallbacks는 1개 결측이며 Image URL이 있으므로 NaN 유지

# 확인
print('\n결측치 처리 후:')
for col in ['Confidence', 'rating', 'Year', 'Gender', 'Price Value', 'Country', 'OilType', 'Price', 'Image Fallbacks', 'Purchase URL']:
    null_n = df[col].isna().sum()
    print(f'  {col}: {null_n}개 ({null_n/len(df)*100:.1f}%)')

처리 전: 101031행
Confidence 결측 제거 후: 101029행

결측치 처리 후:
  Confidence: 0개 (0.0%)
  rating: 13286개 (13.2%)
  Year: 37399개 (37.0%)
  Gender: 0개 (0.0%)
  Price Value: 0개 (0.0%)
  Country: 0개 (0.0%)
  OilType: 0개 (0.0%)
  Price: 43886개 (43.4%)
  Image Fallbacks: 1개 (0.0%)
  Purchase URL: 32344개 (32.0%)


In [6]:
# 이상치 확인 - 주요 카테고리형 컬럼
for col in ['Gender', 'Confidence', 'Popularity', 'Longevity', 'Sillage', 'Price Value', 'Country', 'OilType']:
    print(f'{col}:')
    counts = df[col].value_counts()
    for val, cnt in counts.items():
        print(f'  {val}: {cnt}개 ({cnt/len(df)*100:.1f}%)')
    print()

Gender:
  women: 41683개 (41.3%)
  unisex: 39883개 (39.5%)
  men: 18760개 (18.6%)
  Men: 584개 (0.6%)
  Unknown: 118개 (0.1%)
  Unisex: 1개 (0.0%)

Confidence:
  low: 73685개 (72.9%)
  high: 14768개 (14.6%)
  medium: 12576개 (12.4%)

Popularity:
  Low: 46522개 (46.0%)
  Medium: 23539개 (23.3%)
  Not popular: 19141개 (18.9%)
  High: 8767개 (8.7%)
  Very high: 3060개 (3.0%)

Longevity:
  Moderate: 54558개 (54.0%)
  Long Lasting: 38405개 (38.0%)
  Weak: 3632개 (3.6%)
  Very Long Lasting: 3585개 (3.5%)
  Poor: 849개 (0.8%)

Sillage:
  Moderate: 57791개 (57.2%)
  Strong: 34617개 (34.3%)
  Soft: 5340개 (5.3%)
  Enormous: 3281개 (3.2%)

Price Value:
  good_value: 38513개 (38.1%)
  okay: 26966개 (26.7%)
  unknown: 19140개 (18.9%)
  overpriced: 16410개 (16.2%)

Country:
  Unknown: 70346개 (69.6%)
  France: 11228개 (11.1%)
  USA: 6749개 (6.7%)
  Italy: 4394개 (4.3%)
  UK: 2429개 (2.4%)
  Spain: 1386개 (1.4%)
  UAE: 1099개 (1.1%)
  Germany: 865개 (0.9%)
  Brazil: 764개 (0.8%)
  Netherlands: 304개 (0.3%)
  Canada: 291개 (0.3%)
  Japan

In [7]:
# 표기 통일 (Title Case)
df['Gender'] = df['Gender'].str.title()
df['Confidence'] = df['Confidence'].str.title()
df['Price Value'] = df['Price Value'].str.title()

print('Gender:')
print(df['Gender'].value_counts())

print('\nConfidence:')
print(df['Confidence'].value_counts())

print('\nPrice Value:')
print(df['Price Value'].value_counts())

Gender:
Gender
Women      41683
Unisex     39884
Men        19344
Unknown      118
Name: count, dtype: int64

Confidence:
Confidence
Low       73685
High      14768
Medium    12576
Name: count, dtype: int64

Price Value:
Price Value
Good_Value    38513
Okay          26966
Unknown       19140
Overpriced    16410
Name: count, dtype: int64


In [8]:
# 중복 제거 (_id 기준)
print(f'제거 전: {len(df)}행')

df = df.drop_duplicates(subset=['_id'], keep='first').copy()

print(f'제거 후: {len(df)}행')
print(f'제거된 행: {101029 - len(df)}개')

제거 전: 101029행
제거 후: 38727행
제거된 행: 62302개


In [9]:
# 숫자형 컬럼: object → float64
# Year: 변환 불가한 값은 NaN 유지
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# 카테고리형 컬럼: object → category
cat_cols = ['Gender', 'Confidence', 'Popularity', 'Longevity', 'Sillage',
            'Price Value', 'Country', 'OilType']
for col in cat_cols:
    df[col] = df[col].astype('category')

print('타입 변환 후:')
print(f'  Year: {df["Year"].dtype}', f'(NaN: {df["Year"].isna().sum()}개)')
for col in cat_cols:
    print(f'  {col}: {df[col].dtype}')

타입 변환 후:
  Year: float64 (NaN: 14337개)
  Gender: category
  Confidence: category
  Popularity: category
  Longevity: category
  Sillage: category
  Price Value: category
  Country: category
  OilType: category


In [10]:
# 최종 데이터 현황
print(f'최종 행수: {len(df)}')
print(f'최종 컬럼 수: {len(df.columns)}')

print('\n=== 결측치 현황 ===')
for col in df.columns:
    null_n = df[col].isna().sum()
    if null_n > 0:
        print(f'  {col}: {null_n}개 ({null_n/len(df)*100:.1f}%)')

print('\n=== 주요 컬럼 분포 ===')
for col in ['Gender', 'Confidence', 'Popularity', 'Longevity', 'Sillage', 'Price Value', 'Country', 'OilType']:
    print(f'\n{col}:')
    counts = df[col].value_counts()
    for val, cnt in counts.items():
        print(f'  {val}: {cnt}개 ({cnt/len(df)*100:.1f}%)')

최종 행수: 38727
최종 컬럼 수: 24

=== 결측치 현황 ===
  Year: 14337개 (37.0%)
  rating: 5208개 (13.4%)
  Price: 18416개 (47.6%)
  Image Fallbacks: 1개 (0.0%)
  Purchase URL: 13503개 (34.9%)

=== 주요 컬럼 분포 ===

Gender:
  Women: 15811개 (40.8%)
  Unisex: 15334개 (39.6%)
  Men: 7540개 (19.5%)
  Unknown: 42개 (0.1%)

Confidence:
  Low: 29461개 (76.1%)
  High: 4793개 (12.4%)
  Medium: 4473개 (11.6%)

Popularity:
  Low: 18935개 (48.9%)
  Medium: 8284개 (21.4%)
  Not popular: 7720개 (19.9%)
  High: 2907개 (7.5%)
  Very high: 881개 (2.3%)

Longevity:
  Moderate: 20654개 (53.3%)
  Long Lasting: 14716개 (38.0%)
  Weak: 1517개 (3.9%)
  Very Long Lasting: 1492개 (3.9%)
  Poor: 348개 (0.9%)

Sillage:
  Moderate: 21725개 (56.1%)
  Strong: 13484개 (34.8%)
  Soft: 2139개 (5.5%)
  Enormous: 1379개 (3.6%)

Price Value:
  Good_Value: 14459개 (37.3%)
  Okay: 10553개 (27.2%)
  Unknown: 7719개 (19.9%)
  Overpriced: 5996개 (15.5%)

Country:
  Unknown: 27821개 (71.8%)
  France: 3772개 (9.7%)
  USA: 2053개 (5.3%)
  Italy: 1974개 (5.1%)
  UK: 864개 (2.2%)
  S

In [11]:
# Main Accords / Main Accords Percentage / Notes / General Notes 정규화 및 결측 처리
# 문자열로 저장된 리스트·딕셔너리라 isna()로는 잡히지 않는 결측이 발견됨 (예: "['N/A']", "{}", {'name': 'N/A', ...})
# 파싱 후 실질적으로 값이 없는 행을 찾아 제외하여 처리

import ast

PLACEHOLDER_TOKENS = {'n/a', 'na', 'none', 'null', ''}
DATA_ARTIFACT_TOKENS = {'longevity', 'sillage'}  # 컬럼명이 accord 값으로 잘못 섞여 들어간 오염 토큰


def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return None


def normalize_accords(lst):
    """소문자 정규화 + strip + N/A성 토큰·데이터 오염 토큰 제거"""
    if not isinstance(lst, list):
        return []
    out = []
    for a in lst:
        a_norm = str(a).strip().lower()
        if a_norm and a_norm not in PLACEHOLDER_TOKENS and a_norm not in DATA_ARTIFACT_TOKENS:
            out.append(a_norm)
    return out

def normalize_notes(notes_dict):
    """Notes(Top/Middle/Base) 안의 name을 소문자 정규화 (imageUrl 등 나머지 필드는 유지)"""
    if not isinstance(notes_dict, dict):
        return notes_dict
    out = {}
    for layer in ('Top', 'Middle', 'Base'):
        new_items = []
        for item in notes_dict.get(layer, []):
            if isinstance(item, dict):
                new_item = dict(item)
                new_item['name'] = str(item.get('name', '')).strip().lower()
                new_items.append(new_item)
        out[layer] = new_items
    return out


def normalize_general_notes(lst):
    """General Notes 리스트를 소문자 정규화"""
    if not isinstance(lst, list):
        return []
    return [str(x).strip().lower() for x in lst if str(x).strip()]


def has_valid_accords(normalized_list):
    """정규화 후 유효한 accord가 하나라도 남아있는지"""
    return len(normalized_list) > 0


def has_valid_accord_percentage(d):
    """Main Accords Percentage(강도 딕셔너리)가 비어있지 않은지"""
    return isinstance(d, dict) and len(d) > 0


def has_valid_notes(notes_dict):
    if not isinstance(notes_dict, dict):
        return False
    for layer in ('Top', 'Middle', 'Base'):
        for item in notes_dict.get(layer, []):
            if isinstance(item, dict):
                name = str(item.get('name', '')).strip().lower()
                if name and name not in PLACEHOLDER_TOKENS:
                    return True
    return False


def has_valid_general_notes(lst):
    if not isinstance(lst, list) or len(lst) == 0:
        return False
    return any(str(x).strip().lower() not in PLACEHOLDER_TOKENS for x in lst)


def normalize_accord_percentage(d):
    """Main Accords Percentage 딕셔너리에서도 오염 토큰 키 제거"""
    if not isinstance(d, dict):
        return d
    return {k: v for k, v in d.items() if str(k).strip().lower() not in DATA_ARTIFACT_TOKENS}

In [12]:
# 파싱 및 유효성 플래그 계산
main_accords_parsed = df['Main Accords'].apply(safe_eval)
accord_pct_parsed = df['Main Accords Percentage'].apply(safe_eval)
notes_parsed = df['Notes'].apply(safe_eval)
general_notes_parsed = df['General Notes'].apply(safe_eval)

normalized_accords = main_accords_parsed.apply(normalize_accords)

accord_ok = normalized_accords.apply(has_valid_accords)
accord_pct_ok = accord_pct_parsed.apply(has_valid_accord_percentage)
notes_ok = notes_parsed.apply(has_valid_notes)
general_notes_ok = general_notes_parsed.apply(has_valid_general_notes)

exclude_mask = ~accord_ok | ~accord_pct_ok | ~notes_ok | ~general_notes_ok

print(f'처리 전: {len(df)}행')
print(f'Main Accords 무효: {(~accord_ok).sum()}건')
print(f'Main Accords Percentage 무효: {(~accord_pct_ok).sum()}건')
print(f'Notes 무효: {(~notes_ok).sum()}건')
print(f'General Notes 무효: {(~general_notes_ok).sum()}건')
print(f'제외 대상(합집합): {exclude_mask.sum()}건 ({exclude_mask.sum()/len(df)*100:.2f}%)')

# Main Accords / Notes / General Notes를 정규화된 값으로 교체 + 제외 대상 필터링
df['Main Accords'] = normalized_accords
df['Main Accords Percentage'] = accord_pct_parsed.apply(normalize_accord_percentage)
df['Notes'] = notes_parsed.apply(normalize_notes)
df['General Notes'] = general_notes_parsed.apply(normalize_general_notes)
df = df.loc[~exclude_mask].reset_index(drop=True)

print(f'\n제외 후 최종 행수: {len(df)}행')

처리 전: 38727행
Main Accords 무효: 81건
Main Accords Percentage 무효: 82건
Notes 무효: 177건
General Notes 무효: 155건
제외 대상(합집합): 238건 (0.61%)

제외 후 최종 행수: 38489행


In [13]:
# 전처리 완료 데이터 저장
OUTPUT_PATH = 'data/processed/fragella_processed.csv'

import os
os.makedirs('data/processed', exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUTPUT_PATH}')
print(f'최종 행수: {len(df)}')

저장 완료: data/processed/fragella_processed.csv
최종 행수: 38489
